In [1]:
import requests
from bs4 import BeautifulSoup
import financedatabase as fd
import pandas as pd
import yfinance as yf
import warnings
import time

# Functions

## Scraping

In [2]:
def get_wiki_table_tickers(url, table_id=None, table_class="wikitable", col_index=0):
    """Helper function to scrape ticker symbols from Wikipedia tables using BeautifulSoup.
       Supports negative indices (e.g., -1 for the last column)."""
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')

    if table_id:
        table = soup.find('table', {'id': table_id})
    else:
        tables = soup.find_all('table', {'class': table_class})
        table = None
        for t in tables:
            header_row = t.find('tr')
            if header_row and any(kw in header_row.text.lower() for kw in ['ticker', 'symbol']):
                table = t
                break
        if not table and tables:
            table = tables[0]

    if not table:
        raise ValueError(f"Could not find a valid table at {url}")

    tickers = []
    for row in table.find_all('tr')[1:]:
        cols = row.find_all(['td', 'th'])
        if len(cols) > abs(col_index):
            txt = cols[col_index].text.strip()
            txt = txt.split('[')[0].strip() # Clean reference brackets
            if txt and txt.lower() not in ['ticker', 'symbol', 'company']:
                tickers.append(txt)

    return tickers

## Market Data

In [3]:
def download_market_data(ticker_list, start_date, end_date, price_col='Close', chunk_size=50, sleep_sec=1):
    warnings.filterwarnings('ignore', category=FutureWarning)
    warnings.filterwarnings('ignore', category=DeprecationWarning)
    warnings.filterwarnings('ignore', message='.*inplace.*')

    all_dfs = []

    for i in range(0, len(ticker_list), chunk_size):
        chunk = ticker_list[i:i + chunk_size]
        if not chunk:
            continue

        raw_data = yf.download(chunk, start_date, end_date, auto_adjust=True, progress=True)
        if raw_data.empty:
            continue

        # Extract the specific price column
        prices = raw_data[price_col]
        if isinstance(prices, pd.Series):
            prices = prices.to_frame(name=chunk[0])

        df = prices.reset_index().melt(
            id_vars='Date',
            var_name='Ticker',
            value_name='Price'
        ).dropna(subset=['Price'])

        all_dfs.append(df)

        if sleep_sec > 0 and (i + chunk_size) < len(ticker_list):
            time.sleep(sleep_sec)

    if not all_dfs:
        return pd.DataFrame(columns=['symbol', 'date', 'close_price_loc'])

    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df = combined_df.rename(columns={'Ticker': 'symbol', 'Date': 'date', 'Price': 'close_price_loc'})
    combined_df = combined_df.sort_values(['symbol', 'date']).reset_index(drop=True)

    return combined_df

In [4]:
def concat_asset_class_dfs(df_list):
  df = pd.concat(df_list, ignore_index=True)
  return df

In [5]:
def append_exchange_rates(df, currencies, start_date, end_date):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])

    target_currencies = [c for c in currencies if c != 'CAD']
    ticker_list = [f"{curr}CAD=X" for curr in target_currencies]

    fx_dicts = {}
    if ticker_list:
        fx_df = download_market_data(ticker_list, start_date, end_date)

        for curr in target_currencies:
            ticker = f"{curr}CAD=X"
            curr_subset = fx_df[fx_df['symbol'] == ticker].copy()

            if not curr_subset.empty:
                # Set date as index, ensure it's a DatetimeIndex, and reindex to a complete daily frequency
                curr_subset['date'] = pd.to_datetime(curr_subset['date'])
                curr_subset = curr_subset.set_index('date').asfreq('D')

                # Forward-fill missing values (weekends, holidays)
                curr_subset['close_price_loc'] = curr_subset['close_price_loc'].ffill()

                # Optional: backfill if there are missing values at the very beginning of the series
                curr_subset['close_price_loc'] = curr_subset['close_price_loc'].bfill()

                # Revert index back to a column for dictionary creation
                curr_subset = curr_subset.reset_index()

                fx_dicts[curr] = dict(zip(
                    curr_subset['date'],
                    curr_subset['close_price_loc']
                ))

    df['fx_rate_to_cad'] = 1.0

    for curr in target_currencies:
        if curr in fx_dicts:
            mask = df['currency'] == curr
            df.loc[mask, 'fx_rate_to_cad'] = df.loc[mask, 'date'].map(fx_dicts[curr])

    return df

## Returns

In [6]:
def calculate_returns(df):
  df = df.sort_values(['symbol', 'date'])
  df['close_price_loc'] = df.groupby('symbol')['close_price_loc'].ffill()
  df['close_price_cad'] = df['close_price_loc'] * df['fx_rate_to_cad']

  df['total_return_cad'] = df.groupby('symbol')['close_price_cad'].transform(lambda x : x.pct_change())
  df['total_return_cad_fwd_1d'] = df.groupby('symbol')['total_return_cad'].shift(-1)
  return df

## Universe

In [7]:
def scrape_equity_tickers():
  #Equities
  sp500_raw = get_wiki_table_tickers(
    "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
    table_id="constituents",
    col_index=0
  )
  sp500_tickers = [t.replace('.', '-') for t in sp500_raw]

  cac_raw = get_wiki_table_tickers(
    "https://en.wikipedia.org/wiki/CAC_40",
    table_class="wikitable",
    col_index=3
  )
  cac40_tickers = []
  for t in cac_raw:
      clean_t = t.split()[0].replace('-', '.')
      if not clean_t.endswith('.PA'):
          clean_t += '.PA'
      cac40_tickers.append(clean_t)

  tsx_raw = get_wiki_table_tickers(
    "https://en.wikipedia.org/wiki/S%26P/TSX_Composite_Index",
    table_class="wikitable",
    col_index=0
  )
  tsx_tickers = []
  for t in tsx_raw:
      clean_t = t.split()[0].replace('.', '-')
      if not clean_t.endswith('.TO'):
          clean_t += '.TO'
      tsx_tickers.append(clean_t)

  ticker_list = tsx_tickers + sp500_tickers + cac40_tickers

  return ticker_list


In [25]:
def build_base_universe(equity_tickers, etf_tickers, crypto_tickers, valid_equity_mkt_caps=['Mid Cap', 'Large Cap', 'Mega Cap']):

  #Equities
  equities_raw = fd.Equities().select().reset_index()

  equities = equities_raw[equities_raw['symbol'].isin(equity_tickers)]
  equities = equities.drop_duplicates(subset=['symbol'])

  equities = equities[equities['market_cap'].isin(valid_equity_mkt_caps)]

  equities_filtered = equities[['symbol', 'name', 'currency', 'country', 'sector', 'industry_group', 'industry', 'exchange', 'market', 'mic', 'market_cap']]
  equities_filtered['instrument_type'] = 'Stock'
  equities_filtered['asset_class'] = 'Equity'
  equities_filtered['is_included_universe'] = True


  equities_sub = equities_filtered[['symbol', 'name', 'currency', 'instrument_type', 'asset_class', 'is_included_universe']].copy()
  print(f"EQUITIES | Count : {equities_filtered['name'].count()} | Unique names : {equities_filtered['name'].nunique()}")

  #ETFs
  etfs_raw = fd.ETFs().select().reset_index()
  etfs_filtered = etfs_raw[
    (etfs_raw['delisted'] == False) &
    (etfs_raw['symbol'].isin(etf_tickers))
  ].dropna(subset=['category', 'category_group', 'family'])

  etfs_filtered = etfs_filtered[['symbol', 'name', 'currency', 'category_group', 'category', 'family', 'exchange', 'mic']]
  etfs_filtered['instrument_type'] = 'ETF'
  etfs_filtered['asset_class'] = etfs_filtered['category_group']
  etfs_filtered['is_included_universe'] = True

  etfs_sub = etfs_filtered[['symbol', 'name', 'currency', 'instrument_type', 'asset_class', 'is_included_universe']].copy()
  print(f"ETFs | Count : {etfs_filtered['name'].count()} | Unique names : {etfs_filtered['name'].nunique()}")


  #CRYPTO
  crypto = fd.Cryptos().select(cryptocurrency=crypto_tickers).reset_index()

  crypto_filtered = crypto[crypto['currency'] == 'CAD']
  crypto_filtered = crypto_filtered[['symbol', 'name', 'cryptocurrency', 'currency']]

  crypto_filtered['instrument_type'] = 'Cryptocurrency'
  crypto_filtered['asset_class'] = 'Alternative'
  crypto_filtered['is_included_universe'] = True
  crypto_sub = crypto_filtered[['symbol', 'name', 'currency', 'instrument_type', 'asset_class', 'is_included_universe']].copy()
  print(f"CRYPTOS | Count : {crypto_filtered['name'].count()} | Unique names : {crypto_filtered['name'].nunique()}")


  universe = concat_asset_class_dfs([equities_sub, etfs_sub, crypto_sub])

  ############################
  # SAVE THE THREE _FILTERED
  #
  #
  ############################

  return universe, equities_filtered, etfs_filtered, crypto_filtered

In [8]:
def adjust_universe(criteria: dict, include: bool = True, target_class: str = "all"):
    """
    Adjusts 'is_included_universe' directly inside the filtered sub-tables.

    Parameters:
    - criteria (dict): Column-value pairs to match (e.g., {"sector": "Energy"} or {"symbol": ["AAPL", "MSFT"]})
    - include (bool): True to set matching rows to True (include), False to set matching rows to False (exclude).
    - target_class (str): 'equities', 'etfs', 'crypto', or 'all'
    """
    global equities_filtered, etfs_filtered, crypto_filtered

    tables_map = {
        "equities": equities_filtered,
        "etfs": etfs_filtered,
        "crypto": crypto_filtered
    }

    selected_tables = tables_map.keys() if target_class == "all" else [target_class]

    for name in selected_tables:
        df = tables_map[name]

        # Ensure the column exists
        if "is_included_universe" not in df.columns:
            df["is_included_universe"] = True # Default state if not initialized

        mask = pd.Series(True, index=df.index)
        applied = False

        for col, val in criteria.items():
            if col in df.columns:
                applied = True
                if isinstance(val, list):
                    mask &= df[col].isin(val)
                else:
                    mask &= (df[col] == val)

        if applied:
            # Modify ONLY the matching rows to the specified include state (True or False)
            df.loc[mask, "is_included_universe"] = include
            affected_count = mask.sum()
            print(f"[{name.upper()}] Set {affected_count} rows to `is_included_universe = {include}`.")
        else:
            print(f"[{name.upper()}] Criteria columns not found in this table.")

In [9]:
def build_active_universe() -> str:
    """
    Concatenates all filtered sub-tables where rows have is_included_universe == True,
    keeping only relevant columns, and updates the master universe table.
    """
    global universe, crypto_filtered, etfs_filtered, equities_filtered

    # 1. Collect filtered sub-tables where is_included_universe is True
    active_subsets = []

    for df, name in zip(
        [crypto_filtered, etfs_filtered, equities_filtered],
        ["crypto", "etfs", "equities"]
    ):
        if "is_included_universe" in df.columns:
            subset = df[df["is_included_universe"] == True].copy()
            if not subset.empty:
                active_subsets.append(subset)

    if not active_subsets:
        return "No assets are currently included in the universe. Please adjust your filters."

    # 2. Concatenate them into a unified active universe view
    # Pandas concat will naturally align common columns (like symbol, name, currency, asset_class)
    # and keep table-specific columns (like family or sector) as NaN where they don't apply.
    active_universe = pd.concat(active_subsets, ignore_index=True)

    # 3. Update the master universe table's inclusion flags based on the final active symbols
    active_symbols = active_universe["symbol"].tolist()
    universe["is_included_universe"] = universe["symbol"].isin(active_symbols)

    print(f"Active universe successfully compiled. Total included assets: {len(active_symbols)} / {universe['symbol'].nunique()} total")

    return universe

## Opt Input

In [10]:
def create_opt_inputs(universe, start_date, end_date, user_id='DEFAULT'):
  #fetch base data
  active_universe = universe[universe['is_included_universe'] == True].copy()
  universe_list = active_universe['symbol'].unique().tolist()
  df = download_market_data(universe_list, start_date, end_date)
  df = pd.merge(df, active_universe, on='symbol', how='left')

  #Currencies
  currency_list = active_universe['currency'].unique().tolist()
  df = append_exchange_rates(df, currency_list, start_date, end_date)

  df = calculate_returns(df)

  df['user_id'] = user_id

  return df

# Stocks Prototyping

## Scraping Stocks via Beautifulsoup

### S&P 500

In [11]:
sp500_raw = get_wiki_table_tickers(
    "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
    table_id="constituents",
    col_index=0
)
sp500_tickers = [t.replace('.', '-') for t in sp500_raw]

In [12]:
sp500_tickers[:5]

['MMM', 'AOS', 'ABT', 'ABBV', 'ACN']

### CAC 40

In [13]:
cac_raw = get_wiki_table_tickers(
    "https://en.wikipedia.org/wiki/CAC_40",
    table_class="wikitable",
    col_index=3
)
cac40_tickers = []
for t in cac_raw:
    clean_t = t.split()[0].replace('-', '.')
    if not clean_t.endswith('.PA'):
        clean_t += '.PA'
    cac40_tickers.append(clean_t)

In [14]:
cac40_tickers[:5]

['AC.PA', 'AI.PA', 'AIR.PA', 'MT.AS.PA', 'CS.PA']

### TSX

In [15]:
tsx_raw = get_wiki_table_tickers(
    "https://en.wikipedia.org/wiki/S%26P/TSX_Composite_Index",
    table_class="wikitable",
    col_index=0
)
tsx_tickers = []
for t in tsx_raw:
    clean_t = t.split()[0].replace('.', '-')
    if not clean_t.endswith('.TO'):
        clean_t += '.TO'
    tsx_tickers.append(clean_t)

In [16]:
tsx_tickers[:5]

['VNP.TO', 'ABRA.TO', 'AAV.TO', 'ARE.TO', 'AEM.TO']

In [17]:
ticker_list = tsx_tickers + sp500_tickers + cac40_tickers
ticker_list[:5]
print(len(ticker_list))

763


## Fetching Constituent Data

In [18]:
equities_raw = fd.Equities().select().reset_index()

In [19]:
equities = equities_raw[equities_raw['symbol'].isin(ticker_list)]
equities = equities.drop_duplicates(subset=['symbol'])

## Filtering

In [20]:
market_caps = ['Large Cap', 'Mega Cap']

In [ ]:
equities = equities[equities['market_cap'].isin(market_caps)]

In [ ]:
equities_filtered = equities[['symbol', 'name', 'currency', 'country', 'sector', 'industry_group', 'industry', 'exchange', 'market', 'mic', 'market_cap']]

In [ ]:
print(f"Count : {equities_filtered['name'].count()} | Unique names : {equities_filtered['name'].nunique()}")

Count : 608 | Unique names : 606


In [ ]:
equities_filtered.groupby('market')['symbol'].nunique()

market
Cboe BZX Exchange            1
Euronext Paris              38
NASDAQ Global Select       156
New York Stock Exchange    337
Toronto Stock Exchange      76
Name: symbol, dtype: int64

In [ ]:
equities_filtered.head()

,symbol,name,currency,country,sector,industry_group,industry,exchange,market,mic,market_cap
25417,A,"Agilent Technologies, Inc.",USD,United States,Health Care,"Pharma, Biotech & Life Sciences",Life Science Tools,NYQ,New York Stock Exchange,XNYS,Large Cap
25889,AAPL,Apple Inc.,USD,United States,Information Technology,Hardware & Electronics,Computers & Hardware,NMS,NASDAQ Global Select,XNAS,Mega Cap
26026,ABBV,AbbVie Inc.,USD,United States,Health Care,"Pharma, Biotech & Life Sciences",Pharmaceuticals,NYQ,New York Stock Exchange,XNYS,Mega Cap
26149,ABNB,"Airbnb, Inc. Class A",USD,United States,Consumer Discretionary,Leisure & Consumer Services,Hospitality & Leisure,NMS,NASDAQ Global Select,XNAS,Large Cap
26200,ABT,Abbott Laboratories,USD,United States,Health Care,Health Care Equipment & Providers,Medical Devices & Supplies,NYQ,New York Stock Exchange,XNYS,Large Cap


In [ ]:
equities_filtered.groupby(['exchange', 'sector', 'industry'])['symbol'].nunique()

exchange  sector                  industry                    
BTS       Financials              Capital Markets                 1
NMS       Communication Services  Entertainment                   4
                                  Fixed & Integrated Telecom      1
                                  Internet Media & Services       4
                                  Media                           3
                                                                 ..
TOR       Materials               Packaging                       1
          Utilities               Electric Utilities              3
                                  Gas Utilities                   1
                                  Multi-Utilities                 1
                                  Power Producers & Renewables    1
Name: symbol, Length: 165, dtype: int64

In [ ]:
currencies = equities_filtered['currency'].unique()

In [ ]:
equities_filtered['instrument_type'] = 'Stock'
equities_filtered['asset_class'] = 'Equity'
equities_filtered['is_included_universe'] = True

In [ ]:
equities_filtered.head()

,symbol,name,currency,country,sector,industry_group,industry,exchange,market,mic,market_cap,instrument_type,asset_class,is_included_universe
25417,A,"Agilent Technologies, Inc.",USD,United States,Health Care,"Pharma, Biotech & Life Sciences",Life Science Tools,NYQ,New York Stock Exchange,XNYS,Large Cap,Stock,Equity,True
25889,AAPL,Apple Inc.,USD,United States,Information Technology,Hardware & Electronics,Computers & Hardware,NMS,NASDAQ Global Select,XNAS,Mega Cap,Stock,Equity,True
26026,ABBV,AbbVie Inc.,USD,United States,Health Care,"Pharma, Biotech & Life Sciences",Pharmaceuticals,NYQ,New York Stock Exchange,XNYS,Mega Cap,Stock,Equity,True
26149,ABNB,"Airbnb, Inc. Class A",USD,United States,Consumer Discretionary,Leisure & Consumer Services,Hospitality & Leisure,NMS,NASDAQ Global Select,XNAS,Large Cap,Stock,Equity,True
26200,ABT,Abbott Laboratories,USD,United States,Health Care,Health Care Equipment & Providers,Medical Devices & Supplies,NYQ,New York Stock Exchange,XNYS,Large Cap,Stock,Equity,True


In [ ]:
equities_sub = equities_filtered[['symbol', 'name', 'currency', 'instrument_type', 'asset_class', 'is_included_universe']].copy()

## Saving

In [ ]:
## SAVE TO TABLE

# ETFs Prototyping

## Fetching

In [ ]:
etf_tickers = [
    'ACWX', 'AGZ', 'AIBF', 'AOA', 'AOK', 'AOM', 'AOR', 'AQWA', 'BKF',
    'CLF.TO', 'CLG.TO', 'COMT', 'DACL.TO', 'DACU.TO', 'DAGL.TO',
    'EFA', 'EEM', 'EDIV', 'ILF', 'IOO', 'KORX.TO', 'IYY',
    'VGV.TO', 'VDY.TO', 'SPY', 'IWM', 'NREA.TO',
    'NSGE.TO', 'NTHM.TO'
]

In [ ]:
etfs_raw = fd.ETFs().select().reset_index()

## Filtering

In [ ]:
etfs_filtered = etfs_raw[
    (etfs_raw['delisted'] == False) &
    (etfs_raw['symbol'].isin(etf_tickers))
].dropna(subset=['category', 'category_group', 'family'])

In [ ]:
etfs_filtered = etfs_filtered[['symbol', 'name', 'currency', 'category_group', 'category', 'family', 'exchange', 'mic']]
etfs_filtered['instrument_type'] = 'ETF'
etfs_filtered['asset_class'] = etfs_filtered['category_group']
etfs_filtered['is_included_universe'] = True
etfs_filtered.head()

,symbol,name,currency,category_group,category,family,exchange,mic,instrument_type,asset_class,is_included_universe
2544,ACWX,iShares MSCI ACWI ex U.S. ETF,USD,Equities,Developed Markets,BlackRock Asset Management,NMS,XNAS,ETF,Equities,True
2727,AGZ,iShares Agency Bond ETF,USD,Fixed Income,Government Bonds,BlackRock Asset Management,PCX,ARCX,ETF,Fixed Income,True
2776,AIBF,iShares Future AI Beneficiaries ETF,USD,Equities,Developed Markets,BlackRock Asset Management,PCX,ARCX,ETF,Equities,True
3047,AOA,iShares Core Aggressive Allocation ETF,USD,Equities,Allocation,BlackRock Asset Management,PCX,ARCX,ETF,Equities,True
3050,AOK,iShares Core Conservative Allocation ETF,USD,Equities,Allocation,BlackRock Asset Management,PCX,ARCX,ETF,Equities,True


In [ ]:
etfs_sub = etfs_filtered[['symbol', 'name', 'currency', 'instrument_type', 'asset_class', 'is_included_universe']].copy()

In [ ]:
etfs_filtered['symbol'].nunique()

29

## Saving

In [ ]:
# Save Table Here

# Crypto Prototyping

## Fetching

In [ ]:
crypto = fd.Cryptos().select(cryptocurrency=['BTC', 'ETH']).reset_index()
crypto.head()

,symbol,name,cryptocurrency,currency,summary,exchange,website
0,BTC-AUD,Bitcoin AUD,BTC,AUD,Bitcoin (BTC) is a cryptocurrency. Users are a...,CCC,https://bitcoin.org/
1,BTC-CAD,Bitcoin CAD,BTC,CAD,Bitcoin (BTC) is a cryptocurrency. Users are a...,CCC,https://bitcoin.org/
2,BTC-CNY,Bitcoin CNY,BTC,CNY,Bitcoin (BTC) is a cryptocurrency. Users are a...,CCC,https://bitcoin.org/
3,BTC-ETH,Bitcoin ETH,BTC,ETH,Bitcoin (BTC) is a cryptocurrency. Users are a...,CCC,https://bitcoin.org/
4,BTC-EUR,Bitcoin EUR,BTC,EUR,Bitcoin (BTC) is a cryptocurrency. Users are a...,CCC,https://bitcoin.org/


## Filtering

In [ ]:
crypto_filtered = crypto[crypto['currency'] == 'CAD']
crypto_filtered = crypto_filtered[['symbol', 'name', 'cryptocurrency', 'currency']]
crypto_filtered.head()

,symbol,name,cryptocurrency,currency
1,BTC-CAD,Bitcoin CAD,BTC,CAD
12,ETH-CAD,Ethereum CAD,ETH,CAD


In [ ]:
crypto_filtered['instrument_type'] = 'Cryptocurrency'
crypto_filtered['asset_class'] = 'Alternative'
crypto_filtered['is_included_universe'] = True
crypto_sub = crypto_filtered[['symbol', 'name', 'currency', 'instrument_type', 'asset_class', 'is_included_universe']].copy()

## Saving

In [ ]:
# Save to table

# Create Universe Df Prototyping

In [ ]:
universe = concat_asset_class_dfs([equities_sub, etfs_sub, crypto_sub])

In [ ]:
universe.tail()

,symbol,name,currency,instrument_type,asset_class,is_included_universe
634,SPY,SPDR S&P 500 ETF Trust,USD,ETF,Equities,True
635,VDY.TO,Vanguard FTSE Canadian High Dividend Yield Ind...,CAD,ETF,Equities,True
636,VGV.TO,Vanguard Canadian Government Bond Index ETF,CAD,ETF,Fixed Income,True
637,BTC-CAD,Bitcoin CAD,CAD,Cryptocurrency,Alternative,True
638,ETH-CAD,Ethereum CAD,CAD,Cryptocurrency,Alternative,True


# Create Opt Input Prototyping

## Download Market Data

In [ ]:
start_date = '2026-01-01'
end_date = '2026-11-10'

In [ ]:
universe_list = universe['symbol'].unique().tolist()
df = download_market_data(universe_list, start_date, end_date)

[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  39 of 39 completed


In [ ]:
aa = df.copy()

In [ ]:
df = pd.merge(df, universe, on='symbol', how='left')
df.head()

,date,symbol,close_price_loc,name,currency,instrument_type,asset_class,is_included_universe
0,2026-01-02,A,136.922485,"Agilent Technologies, Inc.",USD,Stock,Equity,True
1,2026-01-05,A,141.865387,"Agilent Technologies, Inc.",USD,Stock,Equity,True
2,2026-01-06,A,146.762466,"Agilent Technologies, Inc.",USD,Stock,Equity,True
3,2026-01-07,A,146.971268,"Agilent Technologies, Inc.",USD,Stock,Equity,True
4,2026-01-08,A,144.932892,"Agilent Technologies, Inc.",USD,Stock,Equity,True


In [ ]:
df['instrument_type'].unique()

<ArrowStringArray>
['Stock', 'ETF', 'Cryptocurrency']
Length: 3, dtype: str

In [ ]:
currency_list = universe['currency'].unique().tolist()
df = append_exchange_rates(df, currency_list, start_date, end_date)
df.tail()

[*********************100%***********************]  2 of 2 completed


,date,symbol,close_price_loc,name,currency,instrument_type,asset_class,is_included_universe,fx_rate_to_cad
136462,2026-11-03,ZTS,82.989998,Zoetis Inc.,USD,Stock,Equity,True,1.39828
136463,2026-11-04,ZTS,89.199997,Zoetis Inc.,USD,Stock,Equity,True,1.40049
136464,2026-11-05,ZTS,82.110001,Zoetis Inc.,USD,Stock,Equity,True,1.39719
136465,2026-11-06,ZTS,79.400002,Zoetis Inc.,USD,Stock,Equity,True,1.40172
136466,2026-11-09,ZTS,82.320000,Zoetis Inc.,USD,Stock,Equity,True,1.41187


In [ ]:
df = calculate_returns(df)

In [ ]:
df

,date,symbol,close_price_loc,name,currency,instrument_type,asset_class,is_included_universe,fx_rate_to_cad,close_price_cad,total_return_cad,total_return_cad_fwd_1d
0,2026-01-02,A,136.922485,"Agilent Technologies, Inc.",USD,Stock,Equity,True,1.37160,187.802885,NaN,0.038419
1,2026-01-05,A,141.865387,"Agilent Technologies, Inc.",USD,Stock,Equity,True,1.37467,195.018096,0.038419,0.036461
2,2026-01-06,A,146.762466,"Agilent Technologies, Inc.",USD,Stock,Equity,True,1.37725,202.128600,0.036461,0.004549
3,2026-01-07,A,146.971268,"Agilent Technologies, Inc.",USD,Stock,Equity,True,1.38155,203.048148,0.004549,-0.010843
4,2026-01-08,A,144.932892,"Agilent Technologies, Inc.",USD,Stock,Equity,True,1.38579,200.846551,-0.010843,0.019391
...,...,...,...,...,...,...,...,...,...,...,...,...
136462,2026-11-03,ZTS,82.989998,Zoetis Inc.,USD,Stock,Equity,True,1.39828,116.043256,-0.001392,0.076527
136463,2026-11-04,ZTS,89.199997,Zoetis Inc.,USD,Stock,Equity,True,1.40049,124.923708,0.076527,-0.081653
136464,2026-11-05,ZTS,82.110001,Zoetis Inc.,USD,Stock,Equity,True,1.39719,114.723270,-0.081653,-0.029869
136465,2026-11-06,ZTS,79.400002,Zoetis Inc.,USD,Stock,Equity,True,1.40172,111.296574,-0.029869,0.044283


# End to End Run (Start to Opt Input)

## Tickers

In [27]:
start_date = '2026-01-01'
end_date = '2026-11-10'

etf_tickers = [
    'ACWX', 'AGZ', 'AIBF', 'AOA', 'AOK', 'AOM', 'AOR', 'AQWA', 'BKF',
    'CLF.TO', 'CLG.TO', 'COMT', 'DACL.TO', 'DACU.TO', 'DAGL.TO',
    'EFA', 'EEM', 'EDIV', 'ILF', 'IOO', 'KORX.TO', 'IYY',
    'VGV.TO', 'VDY.TO', 'SPY', 'IWM', 'NREA.TO',
    'NSGE.TO', 'NTHM.TO'
]

equity_tickers = scrape_equity_tickers()
crypto_tickers = ['BTC', 'ETH']

In [28]:
len(equity_tickers)

763

## Base Universe

In [29]:
universe, equities_filtered, etfs_filtered, crypto_filtered = build_base_universe(equity_tickers, etf_tickers, crypto_tickers)

/tmp/ipykernel_347073/3911008270.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  equities_filtered['instrument_type'] = 'Stock'
/tmp/ipykernel_347073/3911008270.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  equities_filtered['asset_class'] = 'Equity'
/tmp/ipykernel_347073/3911008270.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.

EQUITIES | Count : 503 | Unique names : 501
ETFs | Count : 18 | Unique names : 18


TypeError: select() got an unexpected keyword argument 'cryptocurrency'

In [23]:
equities_filtered.head()

NameError: name 'equities_filtered' is not defined

## Universe Adjustment

In [ ]:
adjust_universe(criteria={"asset_class": "Equity"}, include=True, target_class="equities")
adjust_universe(criteria={"family": "BlackRock Asset Management"}, include=False, target_class="etfs")

[EQUITIES] Set 608 rows to `is_included_universe = True`.
[ETFS] Set 17 rows to `is_included_universe = False`.


In [ ]:
universe = build_active_universe()

Active universe successfully compiled. Total included assets: 622 / 639 total


## Opt Input

In [ ]:

opt = create_opt_inputs(universe, start_date, end_date, 'Ionik')
opt

[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  22 of 22 completed
[*********************100%***********************]  2 of 2 completed


,date,symbol,close_price_loc,name,currency,instrument_type,asset_class,is_included_universe,fx_rate_to_cad,close_price_cad,total_return_cad,total_return_cad_fwd_1d,user_id
0,2026-01-02,A,136.922485,"Agilent Technologies, Inc.",USD,Stock,Equity,True,1.37160,187.802885,NaN,0.038419,Ionik
1,2026-01-05,A,141.865387,"Agilent Technologies, Inc.",USD,Stock,Equity,True,1.37467,195.018096,0.038419,0.036461,Ionik
2,2026-01-06,A,146.762466,"Agilent Technologies, Inc.",USD,Stock,Equity,True,1.37725,202.128600,0.036461,0.004549,Ionik
3,2026-01-07,A,146.971268,"Agilent Technologies, Inc.",USD,Stock,Equity,True,1.38155,203.048148,0.004549,-0.010843,Ionik
4,2026-01-08,A,144.932892,"Agilent Technologies, Inc.",USD,Stock,Equity,True,1.38579,200.846551,-0.010843,0.019391,Ionik
...,...,...,...,...,...,...,...,...,...,...,...,...,...
132988,2026-11-03,ZTS,82.989998,Zoetis Inc.,USD,Stock,Equity,True,1.39828,116.043256,-0.001392,0.076527,Ionik
132989,2026-11-04,ZTS,89.199997,Zoetis Inc.,USD,Stock,Equity,True,1.40049,124.923708,0.076527,-0.081653,Ionik
132990,2026-11-05,ZTS,82.110001,Zoetis Inc.,USD,Stock,Equity,True,1.39719,114.723270,-0.081653,-0.029869,Ionik
132991,2026-11-06,ZTS,79.400002,Zoetis Inc.,USD,Stock,Equity,True,1.40172,111.296574,-0.029869,0.044283,Ionik
